In [ ]:
import json

with open('output/output.json', 'r') as f:
    result = json.load(f)

In [2]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_occur_time(data, pdf_path='num_occur_time.pdf', name2title=None, x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数'):
    # 筛选 type == NUM_OCCUR_TIME
    occur_data = [d for d in data if d['type'] == 'NUM_OCCUR_TIME']
    if name2title is not None:
        # 只保留 name 在 name2title 中的项
        occur_data = [d for d in occur_data if d['name'] in name2title]
        # 断言：要求的 name 在数据中必须全部存在
        present = {d['name'] for d in occur_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(occur_data) > 0, "没有符合条件的数据"

    # 收集所有键并排序
    all_keys = sorted({int(k) for item in occur_data for k in item['value']})
    x_labels = [str(k) for k in all_keys]

    stats_names, stats_values = [], []
    for item in occur_data:
        # 图例使用 name2title 映射，未提供则使用原始 name
        legend_name = name2title[item['name']] if name2title else item['name']
        stats_names.append(legend_name)
        stats_values.append([item['value'].get(str(k), 0) for k in all_keys])

    n_groups = len(all_keys)
    n_stats = len(stats_names)
    bar_width = 0.8 / n_stats
    index = np.arange(n_groups)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for i, (vals, name) in enumerate(zip(stats_values, stats_names)):
        bars = ax.bar(index + i * bar_width, vals, bar_width,
                      label=name, color=colors[i % len(colors)])
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2.,
                        bar.get_height() + max(vals)*0.01,
                        str(val), ha='center', va='bottom',
                        fontsize=6, rotation=90)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(index + bar_width * (n_stats - 1) / 2)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_richi_player_num': '单局立直玩家',
}
plot_num_occur_time(result, 'output/richi_player_num.pdf', name2title=name_map, x_label='本局立直玩家数', y_label='发生次数', title='单局立直玩家数量')


name_map = {
    'stats_richi_ok_num': 'n巡目立直成功玩家数',
    'stats_first_richi_ok_num': 'n巡目先制立直成功玩家数',
    'stats_chasing_richi_ok_num': 'n巡目追立直成功玩家数',
    'stats_be_chased_richi_ok_num': 'n巡目立直但被追立直成功玩家数',
}
plot_num_occur_time(result, 'output/richi_num.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数')

图表已保存至 output/richi_player_num.pdf
图表已保存至 output/richi_num.pdf


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_to_mean_std_sample(data, pdf_path='mean_std_sample.pdf', name2title=None,
                                x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数',
                                offset=0.3):
    """
    将 NUM_TO_MEAN_STD_SAMPLE 数据绘制为带误差棒的折线图并保存为 PDF。
    参数 name2title: 字典，键为原始 name，值为图例显示名称。仅绘制 name 在字典中的项。
    参数 offset: 同一列上不同折线的水平偏移总量，默认0.15。设为0则不偏移。
    """
    # 筛选类型
    mean_std_data = [d for d in data if d['type'] == 'NUM_TO_MEAN_STD_SAMPLE']
    if name2title is not None:
        mean_std_data = [d for d in mean_std_data if d['name'] in name2title]
        present = {d['name'] for d in mean_std_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(mean_std_data) > 0, "没有符合条件的数据"

    # 提取所有巡目并排序
    all_keys_std = sorted({int(k) for item in mean_std_data for k in item['value']})
    x_base = np.array(all_keys_std)          # 原始整数位置
    n_lines = len(mean_std_data)

    # 计算每条线的水平偏移量，使它们均匀分布在基准点左右
    if n_lines > 1 and offset > 0:
        shifts = np.linspace(-offset/2, offset/2, n_lines)
    else:
        shifts = [0.0] * n_lines

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf']
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for idx, item in enumerate(mean_std_data):
        legend_name = name2title[item['name']] if name2title else item['name']
        x_pos = x_base + shifts[idx]          # 偏移后的横坐标
        means, yerrs = [], []
        for k in all_keys_std:
            v = item['value'].get(str(k), None)
            if v is not None:
                means.append(v['mean'])
                sem = v['std'] / np.sqrt(v['total']) if v['total'] > 0 else 0
                yerrs.append(sem)
            else:
                means.append(np.nan)
                yerrs.append(0)

        ax.errorbar(x_pos, means, yerr=yerrs,
                    fmt=markers[idx % len(markers)] + '-',
                    color=colors[idx % len(colors)],
                    label=legend_name,
                    capsize=3, linewidth=1.5, markersize=5)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(x_base)                     # 刻度仍在原始整数位置
    ax.set_xticklabels(all_keys_std)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_richi_n_ron_rate': 'n巡立直荣和率',
    'stats_richi_n_tsumo_rate': 'n巡立直自摸率',
    'stats_richi_n_be_ron_rate': 'n巡立直被荣和率',
    'stats_richi_n_be_tsumo_rate': 'n巡立直被自摸率',
    'stats_richi_n_draw_rate': 'n巡立直横移动率',
    'stats_richi_n_ryuukyoku_rate': 'n巡立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_richi_n_gain': 'n巡立直局收支',
    'stats_richi_n_ron_gain': 'n巡立直荣和局收支',
    'stats_richi_n_tsumo_gain': 'n巡立直自摸局收支',
    'stats_richi_n_be_ron_gain': 'n巡立直被荣和局收支',
    'stats_richi_n_be_tsumo_gain': 'n巡立直被自摸局收支',
    'stats_richi_n_draw_gain': 'n巡立直横移动局收支',
    'stats_richi_n_ryuukyoku_gain': 'n巡立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_first_richi_n_ron_rate': 'n巡先制立直荣和率',
    'stats_first_richi_n_tsumo_rate': 'n巡先制立直自摸率',
    'stats_first_richi_n_be_ron_rate': 'n巡先制立直被荣和率',
    'stats_first_richi_n_be_tsumo_rate': 'n巡先制立直被自摸率',
    'stats_first_richi_n_draw_rate': 'n巡先制立直横移动率',
    'stats_first_richi_n_ryuukyoku_rate': 'n巡先制立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_first_richi_n_gain': 'n巡先制立直局收支',
    'stats_first_richi_n_ron_gain': 'n巡先制立直荣和局收支',
    'stats_first_richi_n_tsumo_gain': 'n巡先制立直自摸局收支',
    'stats_first_richi_n_be_ron_gain': 'n巡先制立直被荣和局收支',
    'stats_first_richi_n_be_tsumo_gain': 'n巡先制立直被自摸局收支',
    'stats_first_richi_n_draw_gain': 'n巡先制立直横移动局收支',
    'stats_first_richi_n_ryuukyoku_gain': 'n巡先制立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_chasing_richi_n_ron_rate': 'n巡追立直荣和率',
    'stats_chasing_richi_n_tsumo_rate': 'n巡追立直自摸率',
    'stats_chasing_richi_n_be_ron_rate': 'n巡追立直被荣和率',
    'stats_chasing_richi_n_be_tsumo_rate': 'n巡追立直被自摸率',
    'stats_chasing_richi_n_draw_rate': 'n巡追立直横移动率',
    'stats_chasing_richi_n_ryuukyoku_rate': 'n巡追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_chasing_richi_n_gain': 'n巡追立直局收支',
    'stats_chasing_richi_n_ron_gain': 'n巡追立直荣和局收支',
    'stats_chasing_richi_n_tsumo_gain': 'n巡追立直自摸局收支',
    'stats_chasing_richi_n_be_ron_gain': 'n巡追立直被荣和局收支',
    'stats_chasing_richi_n_be_tsumo_gain': 'n巡追立直被自摸局收支',
    'stats_chasing_richi_n_draw_gain': 'n巡追立直横移动局收支',
    'stats_chasing_richi_n_ryuukyoku_gain': 'n巡追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_ron_rate': 'n巡被追立直荣和率',
    'stats_be_chased_richi_n_tsumo_rate': 'n巡被追立直自摸率',
    'stats_be_chased_richi_n_be_ron_rate': 'n巡被追立直被荣和率',
    'stats_be_chased_richi_n_be_tsumo_rate': 'n巡被追立直被自摸率',
    'stats_be_chased_richi_n_draw_rate': 'n巡被追立直横移动率',
    'stats_be_chased_richi_n_ryuukyoku_rate': 'n巡被追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_gain': 'n巡被追立直局收支',
    'stats_be_chased_richi_n_ron_gain': 'n巡被追立直荣和局收支',
    'stats_be_chased_richi_n_tsumo_gain': 'n巡被追立直自摸局收支',
    'stats_be_chased_richi_n_be_ron_gain': 'n巡被追立直被荣和局收支',
    'stats_be_chased_richi_n_be_tsumo_gain': 'n巡被追立直被自摸局收支',
    'stats_be_chased_richi_n_draw_gain': 'n巡被追立直横移动局收支',
    'stats_be_chased_richi_n_ryuukyoku_gain': 'n巡被追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')

图表已保存至 output/richi_result_rate.pdf
图表已保存至 output/richi_result_gain.pdf
图表已保存至 output/first_richi_result_rate.pdf
图表已保存至 output/first_richi_result_gain.pdf
图表已保存至 output/chasing_richi_result_rate.pdf
图表已保存至 output/chasing_richi_result_gain.pdf
图表已保存至 output/be_chased_richi_result_rate.pdf
图表已保存至 output/be_chased_richi_result_gain.pdf
